In [1]:
!nvidia-smi

Fri Dec 26 22:19:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   43C    P8             30W /  170W |       1MiB /  12288MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from data_util import OptionDataset
from hyperiv_util import SetEmbeddingNetwork, HyperNetwork
from trainer_util import trainer
import torch.optim as optim

In [3]:
df = pd.read_hdf('spx_w_ref.h5', 'df')

In [4]:
df.is_ref.value_counts()

is_ref
False    1396290
True      349530
Name: count, dtype: int64

In [5]:
N = 1024
B = 128

#train_split = "2023-01-01"
train_split = "2025-12-23"
train_dates = df[df["date"] < train_split]["date"].unique()
train_dataset = OptionDataset(df[df["date"].isin(train_dates)], N=N, sample=True)
train_dataloader = DataLoader(train_dataset, batch_size=B, shuffle=True, drop_last=True)

test_dates = df[df["date"] >= train_split]["date"].unique()
test_dataset = OptionDataset(df[df["date"].isin(test_dates)], N=N, sample=False)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=False)

In [6]:
iv_network = torch.nn.Sequential(
    torch.nn.Linear(2, 16),
    torch.nn.Tanh(),
    torch.nn.Linear(16, 16),
    torch.nn.Tanh(),
    torch.nn.Linear(16, 1),
    torch.nn.Softplus()
)

In [7]:
n_params = sum([p.numel() for p in iv_network.parameters()])
n_params

337

In [8]:
input_dim = 3
output_dim = n_params

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

hyper_model = SetEmbeddingNetwork(input_dim, output_dim).to(device)

model = HyperNetwork(hyper_model, iv_network)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 500
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs, eta_min=1e-5)

cuda


In [ ]:
for epoch in range(num_epochs):
    train_loss_mse, train_loss_mae, train_loss_cal, train_loss_g, train_loss_integral = trainer(train_dataloader, model, device, optimizer, is_train=True)
    print(f'Epoch {epoch+1}/{num_epochs}, Train MSE: {train_loss_mse:.8f}, Train MAE: {train_loss_mae:.8f}, Train CAL: {train_loss_cal:.8f}, Train G: {train_loss_g:.8f}, Train Integral: {train_loss_integral:.8f}')
    lr_scheduler.step()
    test_loss_mse, test_loss_mae, test_loss_cal, test_loss_g, test_loss_integral = trainer(test_dataloader, model, device, optimizer, is_train=False)
    print(f'Epoch {epoch+1}/{num_epochs}, Test MSE: {test_loss_mse:.8f}, Test MAE: {test_loss_mae:.8f}, Test CAL: {test_loss_cal:.8f}, Test G: {test_loss_g:.8f}, Test Integral: {test_loss_integral:.8f}')


00%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.57it/s]

Epoch 1/500, Train MSE: nan, Train MAE: nan, Train CAL: nan, Train G: nan, Train Integral: nan



00%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [00:08<00:00, 68.38it/s]

Epoch 1/500, Test MSE: nan, Test MAE: nan, Test CAL: nan, Test G: nan, Test Integral: nan



00%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.82it/s]

Epoch 2/500, Train MSE: nan, Train MAE: nan, Train CAL: nan, Train G: nan, Train Integral: nan


 58%|████████████████████████████████████████████████████████████████████████████████████████▋                                                               | 350/600 [00:05<00:03, 66.57it/s]

In [ ]:
torch.save(hyper_model.state_dict(), 'spx_hyperiv.pth')